In [1]:
import pandas as pd
import geopandas as gpd

print("1. Loading flood data and creating Point geometry...")
# Load the aggregated sensor data
df_floods = pd.read_parquet('floodnet_floods_only.parquet')

# Aggregate to get max depth and total minutes per sensor
sensor_stats = df_floods.groupby(['deployment_id', 'name', 'latitude', 'longitude']).agg(
    max_depth_inches=('depth_inches', 'max'),
    total_flood_minutes=('depth_inches', 'count') 
).reset_index()

# Convert to GeoDataFrame
gdf_sensors = gpd.GeoDataFrame(
    sensor_stats, 
    geometry=gpd.points_from_xy(sensor_stats.longitude, sensor_stats.latitude),
    crs="EPSG:4326" 
)

print("2. Fetching NYC Borough boundaries...")
nyc_url = "https://data.cityofnewyork.us/resource/gthc-hcne.geojson"
nyc_boroughs = gpd.read_file(nyc_url)

# --- DEBUG STEP: Identify the correct column name ---
# Looking for the column that contains 'Brooklyn', 'Queens', etc.
possible_cols = ['boro_name', 'boroname', 'name', 'label']
boro_col = next((c for c in possible_cols if c in nyc_boroughs.columns), None)

if not boro_col:
    print(f"Available columns: {nyc_boroughs.columns.tolist()}")
    raise KeyError("Could not find a borough name column in the GeoJSON.")

print(f"Found borough column: '{boro_col}'")

# --- Step 3: Align CRS ---
gdf_sensors = gdf_sensors.to_crs("EPSG:2263")
nyc_boroughs = nyc_boroughs.to_crs("EPSG:2263")

print("4. Performing the Spatial Join...")
# Use the boro_col we just identified
joined_data = gpd.sjoin(
    gdf_sensors, 
    nyc_boroughs[[boro_col, 'geometry']], 
    how="left", 
    predicate="within"
)

# Clean up and rename to a standard 'borough' column for the summary
joined_data = joined_data.rename(columns={boro_col: 'borough'})
joined_data = joined_data.drop(columns=['index_right'])

print("\n--- Summary Table ---")
borough_summary = joined_data.groupby('borough').agg(
    total_sensors=('deployment_id', 'count'),
    avg_max_depth_inches=('max_depth_inches', 'mean'),
    max_recorded_depth=('max_depth_inches', 'max')
).round(2).sort_values(by='max_recorded_depth', ascending=False)

print(borough_summary)

# Optional: Save the joined dataset for later use
joined_data.drop(columns='geometry').to_csv('sensors_with_boroughs.csv', index=False)

1. Loading flood data and creating Point geometry...


FileNotFoundError: [Errno 2] No such file or directory: 'floodnet_floods_only.parquet'